# 04 — Model Training

**Project:** Supply Chain Analysis and Prediction  

## Purpose
Train 5 classification models on the consensus feature set to predict supply chain delivery delays.

## Flow

| Step | Description |
|------|-------------|
| 1 | Load selected feature dataset |
| 2 | Train/test split |
| 3 | Train 5 models |
| 4 | Evaluate — Accuracy, F1, AUC-ROC, CV Accuracy |
| 5 | Feature importance (best model) |
| 6 | Save models + results |

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import joblib

from src.config import (
    REPORTS_DIR, IMAGES_DIR, MODELS_DIR, TARGET_COL,
    PLOT_STYLE, FIGURE_DPI, RANDOM_STATE, TEST_SIZE, CV_FOLDS,
    XGBOOST_PARAMS, RF_PARAMS
)

plt.style.use(PLOT_STYLE)
pd.set_option('display.max_columns', None)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Load Selected Feature Dataset

In [2]:
df = pd.read_csv(REPORTS_DIR / 'data_selected_features.csv')
print(f'Shape: {df.shape}')
display(df.head())

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

feature_names = list(X.columns)
print(f'\nFeatures ({len(feature_names)}): {feature_names}')
print(f'Target balance: {y.value_counts().to_dict()}')
print(f'Delay rate: {y.mean()*100:.1f}%')

Shape: (50, 9)


,Cost_Per_Unit,Delivery_DayOfWeek,Delivery_Month,Delivery_Quarter,Delivery_Status,Is_High_Value,Quantity,Shipping_Method,Delay_Label
0,16.02,3,6,2,3,0,45,0,0
1,15.47,4,6,2,3,0,37,2,0
2,41.42,6,6,2,0,1,45,3,1
3,9.60,4,6,2,0,0,53,1,1
4,29.13,0,6,2,0,1,68,0,1



Features (8): ['Cost_Per_Unit', 'Delivery_DayOfWeek', 'Delivery_Month', 'Delivery_Quarter', 'Delivery_Status', 'Is_High_Value', 'Quantity', 'Shipping_Method']
Target balance: {0: 34, 1: 16}
Delay rate: 32.0%


## 2. Train / Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')
print(f'Train delay rate : {y_train.mean()*100:.1f}%')
print(f'Test  delay rate : {y_test.mean()*100:.1f}%')

Train: 40 rows | Test: 10 rows
Train delay rate : 32.5%
Test  delay rate : 30.0%


## 3. Train 5 Models

In [4]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(**RF_PARAMS),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(**XGBOOST_PARAMS, eval_metric='logloss'),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    print(f'Trained: {name}')

Trained: Logistic Regression
Trained: Decision Tree


Trained: Random Forest


Trained: Gradient Boosting
Trained: XGBoost


## 4. Evaluate All Models

In [5]:
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in fitted.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    auc  = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    cv   = cross_val_score(model, X, y, cv=skf, scoring='accuracy').mean()

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1 Score': round(f1, 4),
        'AUC-ROC': round(auc, 4),
        f'CV Accuracy ({CV_FOLDS}-fold)': round(cv, 4),
    })
    print(f'{name:<22} Acc={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | CV={cv:.4f}')

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
print('\n--- Full Results Table ---')
display(results_df)

Logistic Regression    Acc=0.9000 | F1=0.9033 | AUC=1.0000 | CV=0.9600
Decision Tree          Acc=1.0000 | F1=1.0000 | AUC=1.0000 | CV=1.0000


Random Forest          Acc=1.0000 | F1=1.0000 | AUC=1.0000 | CV=1.0000


Gradient Boosting      Acc=1.0000 | F1=1.0000 | AUC=1.0000 | CV=1.0000
XGBoost                Acc=1.0000 | F1=1.0000 | AUC=1.0000 | CV=1.0000

--- Full Results Table ---


,Model,Accuracy,F1 Score,AUC-ROC,CV Accuracy (5-fold)
0,Logistic Regression,0.9,0.9033,1.0,0.96
1,Decision Tree,1.0,1.0000,1.0,1.00
2,Random Forest,1.0,1.0000,1.0,1.00
3,Gradient Boosting,1.0,1.0000,1.0,1.00
4,XGBoost,1.0,1.0000,1.0,1.00


In [6]:
# ── Model comparison bar chart ──
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(results_df))
width = 0.22

ax.bar(x - width,     results_df['Accuracy'],  width, label='Accuracy',  color='steelblue',     edgecolor='white')
ax.bar(x,             results_df['F1 Score'],  width, label='F1 Score',  color='darkorange',    edgecolor='white')
ax.bar(x + width,     results_df['AUC-ROC'],   width, label='AUC-ROC',   color='mediumseagreen',edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right')
ax.set_title('Model Comparison — Accuracy, F1 Score, AUC-ROC')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.legend()

for i, row in results_df.iterrows():
    ax.text(i - width,     row['Accuracy'] + 0.02, f"{row['Accuracy']:.2f}",  ha='center', va='bottom', fontsize=8)
    ax.text(i,             row['F1 Score'] + 0.02, f"{row['F1 Score']:.2f}",  ha='center', va='bottom', fontsize=8)
    ax.text(i + width,     row['AUC-ROC']  + 0.02, f"{row['AUC-ROC']:.2f}",   ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fig.savefig(IMAGES_DIR / '08_model_comparison.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/08_model_comparison.png')

Plot saved -> images/08_model_comparison.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7320\2342206264.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── CV Accuracy comparison ──
cv_col = f'CV Accuracy ({CV_FOLDS}-fold)'
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(results_df['Model'][::-1], results_df[cv_col][::-1],
               color='steelblue', edgecolor='white')
ax.set_xlabel('CV Accuracy')
ax.set_title(f'{CV_FOLDS}-Fold Cross-Validation Accuracy')
ax.set_xlim(0, 1.1)
for bar, val in zip(bars, results_df[cv_col][::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '08b_cv_accuracy.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/08b_cv_accuracy.png')

Plot saved -> images/08b_cv_accuracy.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7320\3262137521.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Feature Importance — Best Model

In [8]:
best_name  = results_df.iloc[0]['Model']
best_model = fitted[best_name]
print(f'Best model by AUC-ROC: {best_name}')

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    imp = np.abs(best_model.coef_[0])
else:
    imp = np.ones(len(feature_names))

imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': imp})\
           .sort_values('Importance', ascending=False).reset_index(drop=True)

print('\nFeature Importances:')
display(imp_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp_df['Feature'][::-1], imp_df['Importance'][::-1], color='steelblue')
ax.set_title(f'Feature Importances — {best_name}')
ax.set_xlabel('Importance')
plt.tight_layout()
fig.savefig(IMAGES_DIR / f'11_feature_importance_{best_name.replace(" ", "_")}.png',
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f'Plot saved -> images/11_feature_importance_{best_name.replace(" ", "_")}.png')

Best model by AUC-ROC: Logistic Regression

Feature Importances:


,Feature,Importance
0,Delivery_Status,2.4774
1,Shipping_Method,0.2415
2,Is_High_Value,0.0870
3,Delivery_DayOfWeek,0.0410
4,Quantity,0.0265
5,Delivery_Month,0.0130
6,Cost_Per_Unit,0.0101
7,Delivery_Quarter,0.0043


Plot saved -> images/11_feature_importance_Logistic_Regression.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_7320\4099919977.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Save Models and Results

In [9]:
# Save all models
for name, model in fitted.items():
    path = MODELS_DIR / f'{name.replace(" ", "_")}.pkl'
    joblib.dump(model, path)
    print(f'Saved: {path.name}')

# Save results
results_path = REPORTS_DIR / 'model_results.csv'
results_df.to_csv(results_path, index=False)
print(f'\nResults saved -> {results_path}')

# Save feature importances
imp_df.to_csv(REPORTS_DIR / 'feature_importance.csv', index=False)
print(f'Feature importance saved -> reports/feature_importance.csv')

Saved: Logistic_Regression.pkl
Saved: Decision_Tree.pkl


Saved: Random_Forest.pkl
Saved: Gradient_Boosting.pkl
Saved: XGBoost.pkl

Results saved -> D:\automation\Supply chain Analysis and Prediction\reports\model_results.csv
Feature importance saved -> reports/feature_importance.csv


## Model Results Summary

| Model | Accuracy | F1 Score | AUC-ROC | CV Accuracy |
|---|---|---|---|---|
| *(filled after execution)* | | | | |

**Key observations:**
- Tree-based models (Random Forest, XGBoost, Gradient Boosting) benefit from the non-linear feature interactions
- `Delivery_Status` is the strongest single predictor — directly encodes delivery outcome
- Logistic Regression provides a strong interpretable baseline
- High accuracy reflects the small dataset (50 rows) — real-world deployment would require more data

---

**Next step:** `05_Model_Evaluation.ipynb` — confusion matrices, ROC curves, residual analysis.